# Self-Representation Exploration

Interactive notebook for exploring results. Run scripts 01-04 first.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from src.dataset import PromptDataset
from src.directions import DirectionResults, ENTITY_CLASS_NAMES
from src.specificity import SpecificityResults
from src.extraction import load_activations
from src.visualization import (
    plot_probe_accuracy, plot_pairwise_similarity,
    plot_entity_projections, plot_confound_similarity,
    plot_control_projections, plot_direction_evolution,
    plot_residual_accuracy,
)

with open('../configs/experiment.yaml') as f:
    cfg = yaml.safe_load(f)

MODEL_NAME = cfg['model']['primary']
MODEL_SLUG = MODEL_NAME.replace('/', '_')
TOKEN_POS = 'final'
DIRS_DIR = f"../data/directions/{MODEL_SLUG}/{TOKEN_POS}"

print(f'Model: {MODEL_NAME}')

In [ ]:
# Load dataset
dataset = PromptDataset.load('../data/prompts.json')
print(f'Total prompts: {len(dataset)}')
print(f'Core prompts: {len(dataset.get_core_prompts())}')

# Show a few examples
for p in dataset.get_core_prompts()[:3]:
    print(f'  [{p.domain} | {p.entity_class}] {p.text[:80]}...')

In [ ]:
# Load direction results (requires script 03 to have run)
dir_results = DirectionResults.load(DIRS_DIR)
print(f'Best probe layer: {dir_results.best_probe_layer}')
print(f'Test accuracy at best layer: {dir_results.probe_test_acc[dir_results.best_probe_layer]:.3f}')
print(f'Avg pairwise cosine sim at best layer: {dir_results.pairwise_avg_cosine_similarity[dir_results.best_probe_layer]:.3f}')

In [ ]:
# Plot probe accuracy
from pathlib import Path
fig_dir = Path('../figures') / MODEL_SLUG / TOKEN_POS
plot_probe_accuracy(dir_results, fig_dir)
plt.show()

In [ ]:
# Entity class projections at best probe layer
plot_entity_projections(dir_results, fig_dir)
plt.show()

In [ ]:
# Pairwise direction similarities
plot_pairwise_similarity(dir_results, fig_dir)
plt.show()

In [ ]:
# Load specificity results (requires script 04 to have run)
spec_results = SpecificityResults.load(DIRS_DIR)
print(f'Best specific layer: {spec_results.best_specific_layer}')
bl = dir_results.best_probe_layer
print(f'cos(self/other, grammatical) at layer {bl}: {spec_results.cos_self_grammatical[bl]:.3f}')
print(f'cos(self/other, animacy) at layer {bl}: {spec_results.cos_self_animacy[bl]:.3f}')
print(f'Residual probe acc at layer {bl}: {spec_results.residual_probe_acc[bl]:.3f}')

In [ ]:
# Confound similarity across layers
plot_confound_similarity(spec_results, fig_dir)
plt.show()

In [ ]:
# Residual vs original probe accuracy
plot_residual_accuracy(spec_results, fig_dir)
plt.show()

In [ ]:
# Direction evolution across layers
plot_direction_evolution(dir_results, fig_dir)
plt.show()

In [ ]:
# Raw activations inspection (loads from disk)
acts_file = f'../data/activations/{MODEL_SLUG}_core.h5'
activations, labels, metadata = load_activations(acts_file, TOKEN_POS)
print(f'Activations shape: {activations.shape}')
print(f'Label distribution: {dict(zip(*np.unique(labels, return_counts=True)))}')

In [ ]:
# Quick PCA of a single layer
from sklearn.decomposition import PCA

layer = dir_results.best_probe_layer
X = activations[:, layer, :]  # (N, D)

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(7, 5))
colors = plt.cm.tab10(np.linspace(0, 0.5, 5))
for i, cls in enumerate(ENTITY_CLASS_NAMES):
    mask = labels == i
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], label=cls, alpha=0.6, s=20, color=colors[i])
ax.legend(fontsize=9)
ax.set_title(f'PCA of Layer {layer} Activations')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.tight_layout()
plt.show()